In [1]:
import ee
import geemap

ee.Authenticate()

ee.Initialize()

print("Google Earth Engine Connected!")

Google Earth Engine Connected!


In [2]:
Map = geemap.Map()

Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [3]:
import ee
import geemap

ee.Initialize(project='ee-chowdharypranav2005')

Map = geemap.Map(center=[19.9975, 73.7898], zoom=7)

Map

Map(center=[19.9975, 73.7898], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [4]:
sentinel = (
    ee.ImageCollection("COPERNICUS/S2_SR")
    .filterDate('2024-01-01', '2024-12-31')
    .filterBounds(ee.Geometry.Point([73.7898, 19.9975]))
    .select(['B4', 'B8'])   # Select only Red and NIR bands
    .median()
)

ndvi = sentinel.normalizedDifference(['B8', 'B4']).rename('NDVI')

In [5]:
ndvi_params = {
    'min': 0,
    'max': 1,
    'palette': ['white', 'yellow', 'green']
}

In [6]:
Map.addLayer(ndvi, ndvi_params, 'NDVI')

Map

Map(bottom=14826.0, center=[19.9975, 73.7898], controls=(WidgetControl(options=['position', 'transparent_bg'],…

In [7]:
sentinel_2020 = (
    ee.ImageCollection("COPERNICUS/S2_SR")
    .filterDate('2020-01-01', '2020-12-31')
    .filterBounds(ee.Geometry.Point([73.7898, 19.9975]))
    .select(['B4', 'B8'])
    .median()
)

ndvi_2020 = sentinel_2020.normalizedDifference(['B8', 'B4']).rename('NDVI_2020')

In [8]:
ndvi_change = ndvi.subtract(ndvi_2020).rename('NDVI_Change')

In [9]:
change_params = {
    'min': -0.5,
    'max': 0.5,
    'palette': ['red', 'white', 'green']
}

In [10]:
Map.addLayer(ndvi_change, change_params, 'NDVI Change 2020-2024')

Map

Map(bottom=14826.0, center=[19.9975, 73.7898], controls=(WidgetControl(options=['position', 'transparent_bg'],…

In [11]:
region = ee.Geometry.Point([73.7898, 19.9975]).buffer(50000)

ndvi_mean = ndvi.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=region,
    scale=30,
    maxPixels=1e9
)

print(ndvi_mean.getInfo())

{'NDVI': 0.21475836438925505}


In [12]:
import ee

ee.Initialize(project='ee-chowdharypranav2005')

image = ee.Image("USGS/SRTMGL1_003")

info = image.getInfo()

print("Success!")

Success!


In [13]:
import ee

ee.Initialize(project='ee-chowdharypranav2005')

point = ee.Geometry.Point([73.7898, 19.9975])

image = (
    ee.ImageCollection("COPERNICUS/S2_SR")
    .filterBounds(point)
    .filterDate('2024-01-01', '2024-12-31')
    .first()
)

print(image.getInfo()['id'])

COPERNICUS/S2_SR/20240104T053221_20240104T053220_T43QCC


In [14]:
import pandas as pd
ndvi_df = pd.DataFrame([ndvi_mean.getInfo()])

ndvi_df.head()

,NDVI
0,0.214758


In [15]:
ndvi_df.to_csv("../data/processed/ndvi_stats.csv", index=False)

print("NDVI stats saved!")

NDVI stats saved!
